In [1]:
import pandas as pd

#lit les clients dtype=str garde le code psotal en texte

customers = pd.read_csv('../data/raw/olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})

# lit les commandes

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')

print(customers.shape, orders.shape)
customers.head()

(99441, 5) (99441, 8)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [2]:
#convertis la date lue comme du texte en vraie date

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

In [3]:
# par customer_id, on relie commande à client

cmd = orders.merge(customers, on='customer_id')
print(len(cmd))

99441


In [4]:
dim_client = (cmd
    .sort_values('order_purchase_timestamp')  # trie par date pour identifier la commande la plus récente
    .drop_duplicates('customer_unique_id', keep='last')  # garde 1 ligne par client (96 096 au lieu de 99 441)
    .reset_index(drop=True) 
    .rename(columns={
        'customer_city': 'ville',
        'customer_zip_code_prefix': 'code_postal_prefix',  # SANS 'e' pour matcher le modèle Django
    })
    [['customer_unique_id', 'ville', 'code_postal_prefix']]
)




In [ ]:
#garde une numeration propre et controle dim_client

assert list(dim_client.columns) == ['customer_unique_id', 'ville', 'code_postal_prefix']
assert len(dim_client) == 96096, f"Attendu 96096, obtenu {len(dim_client)}"
assert dim_client['customer_unique_id'].is_unique
assert dim_client['code_postal_prefix'].str.len().between(4, 5).all()  # les codes postaux brésiliens peuvent avoir 4 ou 5 chiffres
assert dim_client.isna().sum().sum() == 0
print (dim_client)


                     customer_unique_id                 ville  \
0      b7d76e111c89f7ebf14761390f0f7d17             boa vista   
1      4854e9b3feff728c13ee5fc7d1547e92           passo fundo   
2      009b0127b727ab0ba422f6d9604487c7   sao jose dos campos   
3      830d5b7aaa3b6f1e9ad63703bec97d23  sao joaquim da barra   
4      0eb1ee9dba87f5b36b4613a65074337c             sao paulo   
...                                 ...                   ...   
96091  9bb92bebd4cb7511e1a02d5e50bc4655             guarulhos   
96092  634420a0ea42302205032ed44ac7fccc                 pirai   
96093  af5454198a97379394cacf676e1e96cb              registro   
96094  262e1f1e26e92e86375f86840b4ffd63                 picos   
96095  87ab9fec999db8bd5774917de3cdf01c              sorocaba   

      code_postal_prefix  
0                  69309  
1                  99025  
2                  12244  
3                  14600  
4                  02975  
...                  ...  
96091              07176  
960

In [9]:
from pathlib import Path
Path('../data/processed').mkdir(parents=True, exist_ok=True)
dim_client.to_csv('../data/processed/dim_client.csv', index=False, encoding='utf-8')
print("Fichier créé")

Fichier créé
